# LLaMEA Noisy BBOB Thesis Experiment Dashboard

This notebook is used on the **UPB Jupyter Server** to monitor background runs, inspect results, and generate thesis-ready statistics and plots. 

**Note:** To prevent loss of progress from connection drops, the experiment itself runs in the background using `server/scripts/03_run_experiment.sh` (or `04_slurm_submit.sh`) and saves incremental checkpoints to the `results/` folder.

In [ ]:
# Setup paths and imports
import sys
import os
import glob
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

RESULTS_DIR = "../../results"
print("Dashboard environment initialized.")

## 1. Load Experiment Checkpoints
We read all the incremental JSON checkpoint files generated by the background experiment.

In [ ]:
def load_results(results_dir):
    pattern = os.path.join(results_dir, "bbob_*_rep_*.json")
    files = glob.glob(pattern)
    
    records = []
    for filepath in files:
        try:
            with open(filepath, "r") as f:
                data = json.load(f)
                # Remove code string from general tabular view to save space
                data_no_code = {k: v for k, v in data.items() if k != "code"}
                records.append(data_no_code)
        except Exception as e:
            print(f"Warning: Failed to load {filepath}: {e}")
            
    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records)

df = load_results(RESULTS_DIR)
if df.empty:
    print("No results found. The experiment might still be starting, or check the results folder.")
else:
    print(f"Successfully loaded {len(df)} run checkpoint(s).")
    display(df.head())

## 2. Progress Monitor
Check how many runs have been completed vs. what's remaining. This is extremely helpful to watch while the experiment runs in the background.

In [ ]:
if not df.empty:
    # Get total expected runs from configuration
    first_run = df.iloc[0]
    noise_std = first_run['noise_std']
    dim = first_run['dim']
    
    completed_counts = df.groupby('problem_id')['repetition'].count().reset_index(name='completed')
    completed_counts['status'] = completed_counts['completed'].apply(
        lambda x: "DONE" if x >= 5 else f"{x}/5 Completed"
    )
    
    print(f"--- Progress Tracker (Dimension: {dim}, Noise Std: {noise_std}) ---")
    display(completed_counts)
else:
    print("No progress to report yet.")

## 3. Statistical Analysis Table (Thesis Copy-Paste)
Calculate the mean and standard deviation of fitness (AOCC scores) achieved across the repetitions for each problem.

In [ ]:
if not df.empty:
    # Filter out any failed runs if fitness was saved as float('-inf')
    valid_runs = df[df['fitness'] != float('-inf')]
    
    stats = valid_runs.groupby('problem_id')['fitness'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    stats.columns = ['BBOB Problem', 'Mean AOCC', 'Std Dev', 'Best Run', 'Worst Run', 'Repetitions']
    
    print("--- Thesis Performance Statistics ---")
    display(stats.round(4))
else:
    print("No stats available.")

## 4. Plot Performance across BBOB problems
Box plot showing the distribution of the final LLaMEA-evolved algorithm fitness (anytime AOCC score) over the repetitions for each evaluated BBOB problem.

In [ ]:
if not df.empty:
    valid_runs = df[df['fitness'] != float('-inf')].copy()
    valid_runs['problem_id'] = valid_runs['problem_id'].astype(str)
    
    fig = px.box(
        valid_runs,
        x='problem_id',
        y='fitness',
        labels={'problem_id': 'BBOB Problem ID', 'fitness': 'Anytime AOCC score'},
        title='LLaMEA Generated Algorithm Performance Distribution',
        points='all',
        color='problem_id'
    )
    
    fig.update_layout(
        template='plotly_white',
        yaxis_range=[0, 1.05],
        showlegend=False
    )
    
    fig.show()
    
    # Optional: Save figure as PDF or PNG for thesis LaTeX document
    # fig.write_image("bbob_performance_boxplot.pdf")
else:
    print("No data to plot.")

## 5. Inspect Generated Algorithm Code
Retrieve the final Python code generated by LLaMEA for a specific problem and repetition.

In [ ]:
def inspect_code(problem_id, repetition, results_dir=RESULTS_DIR):
    filepath = os.path.join(results_dir, f"bbob_{problem_id}_rep_{repetition}.json")
    if not os.path.exists(filepath):
        print(f"No checkpoint found for Problem {problem_id}, Repetition {repetition}")
        return
        
    with open(filepath, "r") as f:
        data = json.load(f)
        print(f"=== Code for BBOB Problem {problem_id}, Repetition {repetition} (AOCC: {data['fitness']:.4f}) ===\n")
        print(data['code'])

# Example: inspect code for Problem 1, Repetition 0
inspect_code(problem_id=1, repetition=0)